# Calculate Vegetation Indices

---

## Overview
Calculating spectral vegetation indices using `clipped_manifest.csv` per processed scene `.nc`files from `02a_preprocess_data`

1. Index Calculation Function
2. Calculate Spectral Indices 
3. Save Summary Table

| Name | Description | Formula |
| --- | --- | --- | 
| NDVI | Baseline vegetation greenness | (nir - red) / (nir + red) |
| EVI | Enhanced Vegetation Index | (2.5* (nir - red) / (nir + (6)* red - (7.5)* blue +1)) |
| NDWI | Normalized Difference Water Index | (green - nir) / (green + nir) |
| NDAVI | Shallow submerged vegetation | (nir - blue) / (nir + blue) |
| SSI-II | Separating seagrass and sand | (green - red) / (green + red) |
| NDTI | Clarity or Turbidity | (red - green) / (red + green) |

---

In [1]:
# pip install...
!pip install seaborn

## Imports

In [1]:
import os
import gc
import xarray as xr 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

### Load Clipped Manifest

In [3]:
clipped_manifest = pd.read_csv("processed/clipped_manifest.csv", parse_dates=["date"])
clipped_manifest["tile"] = clipped_manifest["scene_id"].str.split("_").str[0]

print(f"Total scenes: {len(clipped_manifest)}")
clipped_manifest.head()

Total scenes: 145


,scene_id,level,year,date,clipped_path,tile
0,T17RNH_20151113T160052,L1C,2015,2015-11-13 16:00:52,processed/clipped\T17RNH_20151113T160052_clipp...,T17RNH
1,T17RNJ_20151113T160052,L1C,2015,2015-11-13 16:00:52,processed/clipped\T17RNJ_20151113T160052_clipp...,T17RNJ
2,T17RNH_20160302T155402,L1C,2016,2016-03-02 15:54:02,processed/clipped\T17RNH_20160302T155402_clipp...,T17RNH
3,T17RNH_20160630T160512,L1C,2016,2016-06-30 16:05:12,processed/clipped\T17RNH_20160630T160512_clipp...,T17RNH
4,T17RNH_20160720T160512,L1C,2016,2016-07-20 16:05:12,processed/clipped\T17RNH_20160720T160512_clipp...,T17RNH


## Index Calculation Function
A predefined function to calculate all spectral indices from a single scene Dataset. Returns a dictionary of spatial mean values per index to append to Dataset. 

In [4]:
def calculate_indices(ds):
    """
    Calculate spectral vegetation and water quality indices
    from Sentinel-2 bands: 
    blue = B02 , green = B03,
    red = B04 , nir = B08.
    Returns dict of mean index values.
    """
    blue, green, red, nir = ds.blue, ds.green, ds.red, ds.nir

    indices = {
        "NDVI": float(((nir - red) / (nir+ red)).mean()),
        "NDWI": float(((green -nir) / (green + nir)).mean()),
        "NDAVI": float(((nir - blue) / (nir + blue)).mean()),
        "EVI": float((2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1)).mean()),
        "SSSII": float(((green - red) / (green +red)).mean()),
        "NDTI": float(((red - green) / (red + green)).mean()),
        # Depth Invariant Index (DII) - green/blue log ratio correction for water column variation
        # log linearizes relationship between reflectance and depth
        "DII": float((np.log(green) - np.log(blue)).mean())
    }

    return indices

## Calculate Indices 
Scene by scene for loop opens each preprocessed `.nc` file with `xr.open_dataset`, calculates all indices using `calculate_indices()`, stores the spatial mean, then closes and frees from memory. Xarray's `open_dataset` was used to avoid coordinate alignment issues during stacking.

In [5]:
# Empty list to store results of index calculations for each scene
summary_data = []

# Iterate through each row in clipped_manifest of clipped & masked scenes
for _, row in clipped_manifest.iterrows():
    # Open individual netCDF files
    ds = xr.open_dataset(row["clipped_path"])

    # Create base dictionary with the Tile ID and timestamp 
    entry = {"tile_id": row["tile"], "time": row["date"]}
    # Call calculate_indices function and append to summary_data
    entry.update(calculate_indices(ds))
    summary_data.append(entry)
    # Memory-first: close ds and manual gc.collect()
    ds.close()
    gc.collect()
    
# Confirmation all scenes indices were calculated
print(f"Processed {len(summary_data)} scenes.")

C:\Users\sulli\miniconda3\envs\myenv\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\sulli\miniconda3\envs\myenv\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\sulli\miniconda3\envs\myenv\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\sulli\miniconda3\envs\myenv\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\sulli\miniconda3\envs\myenv\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\sulli\

Processed 145 scenes.


### Save Summary Table

In [6]:
# Create Pandas DataFrame of indices summary and sort by timestamp
summary_table = pd.DataFrame(summary_data)
summary_table["time"] = (pd.DataFrame(summary_table["time"]))
summary_table = summary_table.sort_values(["tile_id", "time"]).reset_index(drop=True)

# Save indices_summary as csv
summary_table.to_csv("processed/indices_summary.csv", index=False)
print(f"Saved: processed/indices_summary.csv")
summary_table[["tile_id", "time", "NDVI", "NDWI", "NDAVI", "EVI", "SSSII", "NDTI", "DII"]].round(4)

display(summary_table.tail(2))
display(summary_table.head(2))

Saved: processed/indices_summary.csv


C:\Users\sulli\AppData\Local\Temp\ipykernel_24552\1020761392.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  summary_table[["tile_id", "time", "NDVI", "NDWI", "NDAVI", "EVI", "SSSII", "NDTI", "DII"]].round(4)


,tile_id,time,NDVI,NDWI,NDAVI,EVI,SSSII,NDTI,DII
143,T17RNJ,2024-03-30 16:05:11,0.037500,0.065974,-0.044229,NaN,0.106635,-0.106635,NaN
144,T17RNJ,2024-11-15 16:05:11,0.051522,0.064873,-0.042059,NaN,0.120193,-0.120193,NaN


,tile_id,time,NDVI,NDWI,NDAVI,EVI,SSSII,NDTI,DII
0,T17RNH,2015-11-13 16:00:52,-0.028805,0.153598,-0.226168,NaN,0.127616,-0.127616,-0.153930
1,T17RNH,2016-03-02 15:54:02,-0.037907,0.181355,-0.248123,inf,0.147029,-0.147029,-0.142977


### NaN Check
Check for columns with all NaN values before proceeding to comparison and mapping. EVI is designed for terrestrial vegetation and frequently fails over water due to the denominator going to 0.

In [7]:
# Check number of NaNs in each column
nan_summary = summary_table.isna().sum()
print("NaN counts per index:")
print(nan_summary[nan_summary > 0])

NaN counts per index:
NDVI       1
NDWI       1
NDAVI      1
EVI      143
DII       92
dtype: int64


In [8]:
# define the maximum amount of missing data (75% of total scenes)
threshold = len(summary_table) * 0.75

# List comprehension to find columns where the sum of NaNs exceeds threshold
# Often catches indices that fail due to cloud cover or land vs water AOI
all_nan_cols = [col for col in summary_table.columns if summary_table[col].isna().sum() > threshold]

if all_nan_cols:
    # Remove the columns >=75% NaNs
    print(f"\nDropping columns with >75% NaN: {all_nan_cols}")
    summary_table = summary_table.drop(columns=all_nan_cols)
else:
    print("\nNo columns exceed 75% NaN threshold.")
    
# Confirmation of indices remaining
print(f"\nFinal columns: {list(summary_table.columns)}")


Dropping columns with >75% NaN: ['EVI']

Final columns: ['tile_id', 'time', 'NDVI', 'NDWI', 'NDAVI', 'SSSII', 'NDTI', 'DII']


---

## Summary
**Output:** `processed/indices_summary.csv` - Table of processed scenes with spatial mean of each index and DII values 

**Key Findings:**
-	EVI failed across all but one scene as it was designed for terrestrial vegetation and approaches 0 over water, therefore EVI was dropped from `summary_table` and `indice_summary.csv`.
-	NDVI is often insufficient for submerged  vegetation such as seagrass due to high absorption of red light rays in water.

**Next:** `04_spatial_mapping` - to load index summary table, join with water quality data to plot spatial maps over time.

## Resources and references

 - Elshall, A. (2026). *xarray* [Environmental Data Science course notebook] FGCU.
 - Code assistance provided by Kiro AI (AWS, 2025). https://kiro.dev
 - Lai, Y., Zhou, Z., Su, B., Xuanyuan, Z., Tang, J., Yan, J., Liang, W., & Chen, J. (2022, November 11). [*Single underwater image enhancement based on differential attenuation compensation.*](https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2022.1047053/full) Frontiers.
 - Torrez-Perez, J. L., McCullum, A., Beaudry, B., & Cruz, S. (2023, November 2).  [*Spectral indices for land and aquatic applications part 2.*](https://appliedsciences.nasa.gov/sites/default/files/2023-10/Spectral_Indices_Part2.pdf) NASA.
 - David R. Lyzenga. Appl. Opt. 17, 379-383 (1978). [*"Passive remote sensing techniques for mapping water depth and bottom features,"*](https://opg.optica.org/ao/abstract.cfm?URI=ao-17-3-379) 
 
